Transformers Would not load for me if I didn't uninstall and reinstall

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

Pip install transformers

In [ ]:
!pip install transformers

Loading the model directly using the HF api token.

In [1]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

import os
from huggingface_hub import InferenceClient
os.environ["HF_TOKEN"] = #API Token
client = InferenceClient(
    provider="auto",
    api_key=os.environ["HF_TOKEN"],)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


4.1 Sentiment State Extraction API key and Tokenizer

In [10]:
text_example = ["Apple stock rose after strong earnings","Guidance disappointed investors"]

results=[]
for text in text_example:
    r = client.text_classification(
        text,
        model="ProsusAI/finbert",
        top_k=3
    )
    results.append(r)

s_i_scores = []
HD_i_vector =[]

# Si(t) is [1,0,-1]*H(Di(t)). Loop keeps both H(Di(t)) and Si(t)
for r in results:
 scores = {d["label"]: d["score"] for d in r}
 positive = scores["positive"]
 neutral = scores["neutral"]
 negative = scores["negative"]

 score = positive-negative
 HD_i_vector.append([positive,neutral,negative])
 s_i_scores.append(score)


print(HD_i_vector,"\n",s_i_scores)

[[0.9266610145568848, 0.04188281670212746, 0.031456105411052704], [0.06635522842407227, 0.06892205029726028, 0.864722728729248]] 
 [0.8952049091458321, -0.7983675003051758]


4.1 Using the Tokenizer and API key

In [8]:
# Use a pipeline as a high-level helper
from transformers import pipeline
pipe = pipeline("text-classification", model="ProsusAI/finbert", top_k=None)  #top_k=None give pos-nue-neg values


text_example = ["Apple stock rose after strong earnings","Guidance disappointed investors"]


results = pipe(text_example)


s_i_scores = []
HD_i_vector =[]


# Si(t) is [1,0,-1]*H(Di(t)). Loop keeps both H(Di(t)) and Si(t)
#print(results)
for r in results:
 scores = {d["label"]: d["score"] for d in r}
 positive = scores["positive"]
 neutral = scores["neutral"]
 negative = scores["negative"]


 score = positive-negative
 HD_i_vector.append([positive,neutral,negative])
 s_i_scores.append(score)

print(HD_i_vector,"\n",s_i_scores)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[[0.9266611337661743, 0.04188275337219238, 0.03145609796047211], [0.0663551315665245, 0.06892195343971252, 0.8647229075431824]] 
 [0.8952050358057022, -0.7983677759766579]


4.2

In [12]:
import numpy as np

s_off = []
current_sentiment = 0

for sentiment in s_i_scores:
  current_sentiment+=sentiment
  s_off.append(current_sentiment)
print(s_off)
'''
s_off[t] is the cumulative sentiment magnitude at the t-th update
If s_i_scores = [.9,-.6,.4] s_off = [.9,.3,.7]
'''

[0.8952049091458321, 0.09683740884065628]


4.3

In [24]:
'''Code from AI for esimating parameters based on live tweets
from tick.hawkes import HawkesExpKern

model = HawkesExpKern(decays=beta)
model.fit(tweet_timestamps)
This will estimate mu_soc and alpha'''

#Using arbitrary values for parameters
event_times = np.arange(len(s_i_scores))

mu_soc = .2
alpha = .8
beta = 1

lambda_soc= []

for t in event_times:
  excitation = 0

  #Integral
  for ti in event_times[:t]:
    excitation+= alpha*np.exp(-beta*(t-ti))

  intensity = mu_soc + excitation
  lambda_soc.append(intensity)
print(lambda_soc)


#Arbitarily defining Tau and Epsilon
tau = 5
epsilon = 10**-6

s_soc = []

for t in range(len(s_i_scores)):

  lower_bound = max(0,t-tau)

  window_scores = s_i_scores[lower_bound:t+1]

  num=sum(window_scores)
  den=len(window_scores)+epsilon

  s_soc.append(num/den)
print(s_soc)

[0.2, np.float64(0.4943035529371539)]
[0.8952040139418181, 0.04841868021098803]


5.1